In [4]:
import sqlite3
import pandas as pd

import numpy as np
from scipy import stats

con = sqlite3.connect(r'../data/raw/airtel.db')

def q(sql):
    return pd.read_sql(sql, con)

In [5]:
def run_sql_file(path):                   # one sql file
    with open(path, 'r') as f:            # read only
        script = f.read()                 # whole file
    con.executescript(script)             # every statement
    con.commit()                          # save to the db
    print('ran', path)

In [6]:
def pval(z, tail='two'):                       # p from a z value
    if tail == 'greater':                      # H1: bigger
        return stats.norm.sf(z)                # area to the right
    if tail == 'less':                         # H1: smaller
        return stats.norm.cdf(z)               # area to the left
    return 2 * stats.norm.sf(abs(z))           # both sides
 
def f_test(x, y):                              # two variances
    f = np.var(x, ddof=1) / np.var(y, ddof=1)  # ratio of spreads
    d1, d2 = len(x) - 1, len(y) - 1            # degrees of freedom
    lo = stats.f.cdf(f, d1, d2)                # area to the left
    p = 2 * min(lo, stats.f.sf(f, d1, d2))     # two-tailed
    print(f'F = {f:.4f}  p = {p:.4g}')         # the result
 
def z_mean2(x, y, tail='two'):                 # two means, large
    se = np.sqrt(np.var(x, ddof=1) / len(x)    # standard error,
                 + np.var(y, ddof=1) / len(y)) # each group its own
    z = (np.mean(x) - np.mean(y)) / se         # gap, in se units
    print(f'z = {z:.4f}  p = {pval(z, tail):.4g}')  # the result

def z_prop2(k1, n1, k2, n2, tail='two'):       # two percentages
    pool = (k1 + k2) / (n1 + n2)               # both groups as one
    se = np.sqrt(pool * (1 - pool) * (1/n1 + 1/n2))  # spread if H0
    z = (k1/n1 - k2/n2) / se                   # gap, in se units
    print(f'z = {z:.4f}  p = {pval(z, tail):.4g}')  # the result

In [9]:
# Claim :the two groups used the same amount of data before the offer 
# H₀ :both groups have the same true average daily data before May
# H₁ :the two averages differ
# Tail :two-tailed — a gap either way would break the split

In [10]:
pre = q("""select c.ab_group as g, u.customer_id, avg(u.data_mb) as m 
from usage_daily u
join customer_features c on c.customer_id = u.customer_id
where u.usage_date < '2025-05-01'
group by u.customer_id;""")

pre

,g,customer_id,m
0,B,1,729.705882
1,A,2,256.590909
2,A,3,1914.133333
3,B,4,218.880000
4,B,5,980.926667
...,...,...,...
21988,A,21996,6681.156863
21989,A,21997,1460.978571
21990,B,21998,69.021429
21991,A,21999,1153.977778


In [12]:
a = pre[pre.g=='A'].m
b = pre[pre.g=='B'].m

f_test(a, b)

z_mean2(a, b)

F = 1.1108  p = 3.673e-08
z = 0.5191  p = 0.6037


In [13]:
cf=q("""
SELECT * FROM customer_features;

""")
A,B=cf[cf.ab_group== 'A'],cf[cf.ab_group== 'B']

for col in ['stated_arpu','tenure_days','complaint_count']:
    print(col)
    f_test(A[col],B[col])

stated_arpu
F = 1.0668  p = 0.0006932
tenure_days
F = 1.0244  p = 0.2061
complaint_count
F = 1.0240  p = 0.2128


In [14]:
rev = q("""
SELECT
    c.ab_group AS g,
    COALESCE(SUM(r.amount_inr), 0) AS s
FROM customer_features c
LEFT JOIN recharges r
    ON r.customer_id = c.customer_id
    AND r.recharge_date >= '2025-01-01'
    AND r.recharge_date < '2025-05-01'
GROUP BY c.customer_id;
""")
rev

,g,s
0,B,897
1,A,198
2,A,3408
3,B,0
4,B,1438
...,...,...
21995,A,6891
21996,A,719
21997,B,836
21998,A,798


In [16]:
A = rev[rev.g=='A'].s
B = rev[rev.g=='B'].s

f_test(B, A)

z_mean2(B, A)

F = 0.9513  p = 0.008866
z = -1.3490  p = 0.1773


In [18]:
q("""select ab_group, count(*) as n,
sum(device_type = 'Smartphone') as smart,
sum(connection_type = 'Prepaid') as prepaid
from customer_features group by ab_group;""")

,ab_group,n,smart,prepaid
0,A,10996,9417,8575
1,B,11004,9554,8574


In [20]:
z_prop2(9554,11004,9417,10996)
z_prop2(8574,11004,8575,10996)

z = 2.5457  p = 0.01091
z = -0.1177  p = 0.9063


In [21]:
# Claim : the offer raised daily data usage
# H₀ :both groups have the same true average daily data in May and June
# H₁ :group B's average is higher
# Tail :one-tailed — the company said so in advance

off=q("""SELECT c.ab_group AS g, u.customer_id,
       AVG(u.data_mb) AS m
FROM usage_daily u
JOIN customer_features c ON c.customer_id = u.customer_id
WHERE u.usage_date >= '2025-05-01'
GROUP BY u.customer_id;""")
off

,g,customer_id,m
0,A,3,1758.756250
1,B,5,1699.688889
2,A,6,339.763333
3,A,7,5363.175000
4,B,8,1376.940000
...,...,...,...
18443,B,21993,1936.235714
18444,A,21996,6483.050000
18445,A,21997,1262.429032
18446,B,21998,86.217391


In [23]:
A = off[off.g=='A'].m
B = off[off.g=='B'].m

A, B

(0        1758.756250
 2         339.763333
 3        5363.175000
 9        3828.633333
 10        580.526316
             ...     
 18439    1132.084375
 18440     900.400000
 18444    6483.050000
 18445    1262.429032
 18447    1588.607500
 Name: m, Length: 9247, dtype: float64,
 1        1699.688889
 4        1376.940000
 5        1865.279310
 6        1216.876316
 7        1041.106667
             ...     
 18436    1049.761905
 18441    1922.758621
 18442    1033.139130
 18443    1936.235714
 18446      86.217391
 Name: m, Length: 9201, dtype: float64)

In [25]:
A = off[off.g=='A'].m
B = off[off.g=='B'].m

f_test(A,B)
# p-value < 0.05 
z_mean2(A,B)

F = 0.6197  p = 8.402e-116
z = -19.4496  p = 2.936e-84


In [27]:
q("""select ab_group, count(*) as n, sum(is_churned) as all_churn, sum(churn_date >= '2025-05-01') as may_on
from customer_features 
group by ab_group;""")

,ab_group,n,all_churn,may_on
0,A,10996,2587,905
1,B,11004,2639,914


In [29]:
rev = q("""select c.ab_group as g, coalesce(sum(r.amount_inr),0) as s
from customer_features c 
left join recharges r on r.customer_id = c.customer_id and r.recharge_date >= '2025-05-01' 
group by c.customer_id;""")

rev

,g,s
0,B,0
1,A,0
2,A,1665
3,B,0
4,B,0
...,...,...
21995,A,3587
21996,A,0
21997,B,209
21998,A,0


In [31]:
a = rev[rev.g=='A'].s
b = rev[rev.g=='B'].s
a, b

(1           0
 2        1665
 5         396
 6        3576
 13       3529
          ... 
 21989       0
 21995    3587
 21996       0
 21998       0
 21999       0
 Name: s, Length: 10996, dtype: int64,
 0          0
 3          0
 4          0
 7          0
 8        349
         ... 
 21991    992
 21992    349
 21993      0
 21994      0
 21997    209
 Name: s, Length: 11004, dtype: int64)

In [32]:
f_test(b, a)

# p-value > 0.05 Hence
# we will go fot ttest

F = 0.9697  p = 0.1068


In [33]:
stats.ttest_ind(b, a, alternative='greater')

TtestResult(statistic=np.float64(-0.5926646653701372), pvalue=np.float64(0.7232941573761509), df=np.float64(21998.0))

In [34]:
q("""select connection_type as grp, count(*) as n, sum(is_churned) as k
from customer_features
group by connection_type;""")

z_prop2(4441, 17149, 785, 4851)
# p-value < 0.05 so its significant
# HENCE REJECT h0

z = 14.0364  p = 9.333e-45


In [35]:
q("""select case when complaint_count = 0 then 'never'
when unresolved_count = 0 then 'all closed'
else 'open' end as grp,
count(*) as n, sum(is_churned) as k
from customer_features
group by grp;""")

,grp,n,k
0,all closed,8689,2064
1,never,6631,1328
2,open,6680,1834


In [36]:
q("""select case when complaint_count = 0 then 'never'
when unresolved_count = 0 then 'all closed'
else 'open' end as grp,
count(*) as n, sum(is_churned) as k
from customer_features
group by grp;""")

z_prop2(1834, 6680, 1328, 6631)
# p-value <  so result is significant
# HENCE REJECT h0

z = 10.0684  p = 7.623e-24


In [41]:
q("""select tenure_years as grp,
count(*) as n, sum(is_churned) as k
from customer_features
group by tenure_years;""")

,grp,n,k
0,0,4610,1328
1,1,6961,1725
2,2,7055,1541
3,3,3374,632


In [39]:

# Claim :year-0 and year-3 subscribers churn at different rates
# H₀ :they share one true churn rate
# H₁ :they do not
q("""
SELECT tenure_years AS grp,
COUNT(*) AS n, SUM(is_churned) AS k
FROM customer_features
GROUP BY tenure_years;
""")
(1328,4610,632,3374)
# p-value<0.05 soz_prop2 its significant
# hence reject ho

(1328, 4610, 632, 3374)

In [40]:
# Claim :Himachal Pradesh and Punjab churn at different rates
# H₀ :both have the same true churn rate
# H₁ :the two rates differ

q("""
SELECT circle_name AS grp,
COUNT(*) AS n, SUM(is_churned) AS k
FROM customer_features
GROUP BY circle_name;
""")
z_prop2(417,1247,77,558)

z = 8.6494  p = 5.176e-18


In [42]:
q("""select device_type as grp,
count(*) as n, sum(is_churned) as k
from customer_features
group by device_type;""")

z_prop2(4507, 18971, 719, 3029)

z = 0.0241  p = 0.9807


In [43]:
q("""select c.gender, c.device_type,
u.data_mb as m
from usage_daily u
join customer_features c on c.customer_id = u.customer_id;""")

,gender,device_type,m
0,Male,Smartphone,213.7
1,Male,Smartphone,1541.0
2,Male,Smartphone,107.4
3,Male,Smartphone,1569.6
4,Male,Smartphone,845.9
...,...,...,...
1326734,Male,Smartphone,335.9
1326735,Male,Smartphone,2465.5
1326736,Male,Smartphone,405.0
1326737,Male,Smartphone,1192.0


In [46]:
# Claim :gender and device type each change the data a subscriber uses
# H₀ :both groups have the same true average daily data
# H₁ :the two averages differ

gen = q("""
SELECT c.gender, c.device_type,
u.data_mb AS m
FROM usage_daily u
JOIN customer_features c ON c.customer_id = u.customer_id;
""")
dev = q("""
SELECT c.gender, c.device_type,
AVG(u.data_mb) AS m
from usage_daily u
JOIN customer_features c ON c.customer_id = u.customer_id
GROUP BY c.customer_id;
""")

In [47]:
male = gen[gen.gender=='Male'].m
female = gen[gen.gender=='Female'].m
female

f_test(female, male)
z_mean2(female, male)

F = 0.9315  p = 6.262e-177
z = -7.2519  p = 4.109e-13


In [48]:
fp = dev[dev.device_type=='Feature Phone'].m
sp = dev[dev.device_type=='Smartphone'].m
f_test(fp, sp)

F = 0.0268  p = 0


In [49]:
# Claim: Subscribers who use roaming churn at a different rate from subscribers who never use roaming.
# H₀: Both groups have the same true churn rate.
# H₁: The two rates differ.

q("""
SELECT
    CASE
        WHEN roaming_days > 0 THEN 'roaming users'
        ELSE 'never roaming'
    END AS grp,
    COUNT(*) AS n,
    SUM(is_churned) AS k
FROM customer_features
GROUP BY grp;
""")

,grp,n,k
0,never roaming,5767,2573
1,roaming users,16233,2653


In [50]:
z_prop2(2653, 16233, 2573, 5767)

z = -43.3361  p = 0


In [51]:
# Claim: Male and female subscribers have different average six-month recharge amounts.
# H₀: Both groups have the same true average recharge amount.
# H₁: The two averages differ.

recharge = q("""
SELECT
    gender,
    recharge_total_6m AS recharge
FROM customer_features
WHERE gender IN ('Male', 'Female');
""")

In [52]:
male = recharge[recharge.gender == 'Male'].recharge
female = recharge[recharge.gender == 'Female'].recharge


f_test(male, female)
z_mean2(male, female)

F = 1.0202  p = 0.3036
z = 0.2324  p = 0.8162


In [53]:
# Claim: Smartphone and feature-phone users have different average daily voice usage.
# H₀: Both groups have the same true average daily voice usage.
# H₁: The two averages differ.

voice = q("""
SELECT
    c.device_type AS grp,
    u.customer_id,
    AVG(u.voice_minutes) AS avg_daily_voice
FROM usage_daily u
JOIN customer_features c
    ON c.customer_id = u.customer_id
WHERE c.device_type IN ('Smartphone', 'Feature Phone')
GROUP BY c.customer_id, c.device_type;
""")

In [54]:
fp = voice[voice.grp == 'Feature Phone'].avg_daily_voice
sp = voice[voice.grp == 'Smartphone'].avg_daily_voice

f_test(fp, sp)
z_mean2(fp, sp)

F = 1.0115  p = 0.6731
z = 0.4166  p = 0.677
